# Phase 10: Feature Scaling

## Objectives
- Load the verified training and testing datasets.
- Identify numerical and categorical features.
- Apply StandardScaler to numerical features.
- Fit the scaler on training data only.
- Transform training and test data consistently.
- Save scaled datasets and fitted scalers.

## Research safeguard
The test set must never be used to fit the scaler.

In [1]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler

current_dir = Path.cwd()

if (current_dir / "data" / "processed" / "splits").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "processed" / "splits").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot find data/processed/splits. "
        "Check the project root."
    )

split_dir = project_root / "data" / "processed" / "splits"
output_dir = project_root / "data" / "processed" / "scaled"
model_dir = project_root / "models" / "scalers"

output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Split directory:", split_dir)
print("Scaled output directory:", output_dir)

Project root: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction
Split directory: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\splits
Scaled output directory: e:\PROJECTS\Mini Project\Project\Student-Performance-Prediction\data\processed\scaled


In [2]:
datasets = {}

for subject in ["mat", "por"]:
    datasets[subject] = {
        "X_train": pd.read_csv(split_dir / f"{subject}_X_train.csv"),
        "X_test": pd.read_csv(split_dir / f"{subject}_X_test.csv"),
        "y_train": pd.read_csv(split_dir / f"{subject}_y_train.csv"),
        "y_test": pd.read_csv(split_dir / f"{subject}_y_test.csv"),
    }

    train = datasets[subject]["X_train"]
    test = datasets[subject]["X_test"]

    assert list(train.columns) == list(test.columns)
    assert len(train) == len(datasets[subject]["y_train"])
    assert len(test) == len(datasets[subject]["y_test"])

    print(f"\n{subject.upper()}")
    print("Training shape:", train.shape)
    print("Testing shape:", test.shape)
    print("Data types:")
    print(train.dtypes.value_counts())


MAT
Training shape: (316, 30)
Testing shape: (79, 30)
Data types:
object    17
int64     13
Name: count, dtype: int64

POR
Training shape: (519, 30)
Testing shape: (130, 30)
Data types:
object    17
int64     13
Name: count, dtype: int64


In [3]:
scaled_data = {}

for subject, data in datasets.items():
    X_train = data["X_train"].copy()
    X_test = data["X_test"].copy()

    numeric_cols = X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_cols = [
        col for col in X_train.columns
        if col not in numeric_cols
    ]

    if not numeric_cols:
        raise ValueError(
            f"No numerical columns found for {subject}"
        )

    # Fit using training data only.
    scaler = StandardScaler()
    X_train.loc[:, numeric_cols] = scaler.fit_transform(
        X_train[numeric_cols]
    )

    # Apply the training-fitted scaler to the test set.
    X_test.loc[:, numeric_cols] = scaler.transform(
        X_test[numeric_cols]
    )

    scaled_data[subject] = {
        "X_train": X_train,
        "X_test": X_test,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "scaler": scaler,
    }

    print(f"\n{subject.upper()}")
    print("Scaled numerical columns:", numeric_cols)
    print("Categorical columns left unchanged:", categorical_cols)
    print("Training numerical means (approximately zero):")
    print(X_train[numeric_cols].mean().round(3))



MAT
Scaled numerical columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Categorical columns left unchanged: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Training numerical means (approximately zero):
age           0.0
Medu          0.0
Fedu          0.0
traveltime    0.0
studytime    -0.0
failures      0.0
famrel        0.0
freetime     -0.0
goout         0.0
Dalc         -0.0
Walc         -0.0
health        0.0
absences     -0.0
dtype: float64

POR
Scaled numerical columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Categorical columns left unchanged: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activi

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7516\3767889325.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.30533216 -0.52411064 -1.30533216  1.81955392  1.0383324  -0.52411064
  1.0383324   1.0383324  -0.52411064 -1.30533216  0.25711088  1.81955392
  1.0383324   1.81955392  0.25711088 -1.30533216 -1.30533216 -1.30533216
  2.60077543 -0.52411064  0.25711088  1.0383324  -0.52411064 -0.52411064
  1.0383324  -0.52411064  1.0383324  -0.52411064 -0.52411064  0.25711088
 -1.30533216 -0.52411064 -1.30533216  0.25711088  1.81955392 -0.52411064
  0.25711088  1.0383324  -0.52411064  0.25711088 -0.52411064  0.25711088
  1.0383324   0.25711088 -1.30533216  0.25711088 -1.30533216 -1.30533216
 -1.30533216 -1.30533216  0.25711088  0.25711088  0.25711088 -0.52411064
  0.25711088  0.25711088 -0.52411064  0.25711088  0.25711088 -1.30533216
 -0.52411064  1.0383324   1.0383324   0.25711088  2.60077543 -0.52411064
 -1.3053321

In [4]:
for subject, data in scaled_data.items():
    data["X_train"].to_csv(
        output_dir / f"{subject}_X_train_scaled.csv",
        index=False,
    )

    data["X_test"].to_csv(
        output_dir / f"{subject}_X_test_scaled.csv",
        index=False,
    )

    joblib.dump(
        data["scaler"],
        model_dir / f"{subject}_standard_scaler.joblib",
    )

    print(f"{subject.upper()}: scaled data and scaler saved.")

MAT: scaled data and scaler saved.
POR: scaled data and scaler saved.


In [5]:
for subject, data in scaled_data.items():
    train_path = output_dir / f"{subject}_X_train_scaled.csv"
    test_path = output_dir / f"{subject}_X_test_scaled.csv"
    scaler_path = model_dir / f"{subject}_standard_scaler.joblib"

    assert train_path.exists()
    assert test_path.exists()
    assert scaler_path.exists()

    train_saved = pd.read_csv(train_path)
    test_saved = pd.read_csv(test_path)

    assert train_saved.shape == data["X_train"].shape
    assert test_saved.shape == data["X_test"].shape
    assert list(train_saved.columns) == list(test_saved.columns)

    print(f"{subject.upper()}: scaling verification passed.")

print("\nFeature scaling completed successfully.")

MAT: scaling verification passed.
POR: scaling verification passed.

Feature scaling completed successfully.
